# 02 Train / validation / test splits

Defines the splits once, before any modelling, and saves them to `splits/splits.json`. its very important to not re-run with another seed after models have been trained !!

- Splits are made at the patient level. la_005 and la_026 are the same scan (see notebook 01), so they form one patient.
- That duplicated patient is kept in the development set, so the test set has one reference segmentation per patient.
- Test set is 5 patients. Development set is 14 patients (15 volumes), divided into 5 cross-validation folds.

In [ ]:
import json
from pathlib import Path

import numpy as np

DATA_ROOT = Path("../data/Task02_Heart")
OUT_FILE = Path("../splits/splits.json")
SEED = 42  # fixed random seed, the splits are always the same (its a seed used in many projects, so that we can compare results with other projects)
N_TEST = 5  # number of test patients
N_FOLDS = 5  # number of cross-validation folds

# names of the 20 labelled volumes (la_003, la_004, ...)
meta = json.loads((DATA_ROOT / "dataset.json").read_text())
cases = sorted(Path(p["image"]).name.replace(".nii.gz", "") for p in meta["training"])

# one patient = one group. la_026 is the same image as la_005
patient_of = {c: c for c in cases}
patient_of["la_026"] = "la_005"
patients = sorted(set(patient_of.values()))
print(f"{len(cases)} volumes, {len(patients)} patients")

20 volumes, 19 patients


In [ ]:
rng = np.random.default_rng(SEED)

# test patients are drawn among patients with a single segmentation
candidates = [p for p in patients if p != "la_005"]
test_patients = sorted(rng.choice(candidates, size=N_TEST, replace=False).tolist())
dev_patients = [p for p in patients if p not in test_patients]

# cross-validation folds: dev patients are shuffled, then dealt one by one into the 5 folds
shuffled = rng.permutation(dev_patients).tolist()
fold_patients = [sorted(shuffled[k::N_FOLDS]) for k in range(N_FOLDS)]

def volumes_of(patient_list):
    # volumes of these patients (patient la_005 gives la_005 and la_026)
    return sorted(c for c in cases if patient_of[c] in patient_list)

splits = {
    "seed": SEED,
    "test": volumes_of(test_patients),
    "dev": volumes_of(dev_patients),
    "folds": [volumes_of(f) for f in fold_patients],  # validation volumes of each fold
}

# sanity checks, see if, no patient in two sets, every dev volume in exactly one fold
assert not set(splits["test"]) & set(splits["dev"])
assert sorted(sum(splits["folds"], [])) == splits["dev"]
assert {"la_005", "la_026"} <= set(splits["dev"])

print("Test: ", splits["test"])
print("Dev:  ", splits["dev"])
for k, f in enumerate(splits["folds"]):
    print(f"Fold {k} validation:", f)

Test:  ['la_004', 'la_016', 'la_019', 'la_020', 'la_030']
Dev:   ['la_003', 'la_005', 'la_007', 'la_009', 'la_010', 'la_011', 'la_014', 'la_017', 'la_018', 'la_021', 'la_022', 'la_023', 'la_024', 'la_026', 'la_029']
Fold 0 validation: ['la_003', 'la_010', 'la_023']
Fold 1 validation: ['la_005', 'la_017', 'la_022', 'la_026']
Fold 2 validation: ['la_007', 'la_014', 'la_024']
Fold 3 validation: ['la_011', 'la_018', 'la_021']
Fold 4 validation: ['la_009', 'la_029']


In [ ]:
# the splits are saved once and never changed afterwards (the models are trained on them globally, so we cannot change them later)
OUT_FILE.parent.mkdir(exist_ok=True)
if OUT_FILE.exists():
    assert json.loads(OUT_FILE.read_text()) == splits, "splits.json already exists and differs: do not overwrite it"
    print("splits.json already exists and is identical")
else:
    OUT_FILE.write_text(json.dumps(splits, indent=2))
    print("Saved to", OUT_FILE)

Saved to ..\splits\splits.json
